# AI5707 — Lab 5: Maximum Likelihood Parameter Estimation

**Objective:** Estimate parameters using Maximum Likelihood Estimation (MLE) on data generated in the previous assignment and on real data.

Assume
\[
t_n=\mathbf w^T\phi(x_n)+\epsilon_n,\qquad
\epsilon_n\sim\mathcal N(0,\beta^{-1}).
\]

For Gaussian noise,
\[
\mathbf w_{ML}=(\Phi^T\Phi)^{-1}\Phi^T\mathbf t,
\qquad
\beta_{ML}=\frac{N}{\sum_n(t_n-\hat t_n)^2}.
\]
Hence
\[
\sigma^2_{ML}=\frac{1}{\beta_{ML}}=\frac{\mathrm{SSE}}{N}.
\]

MLE of the regression weights is therefore equivalent to least squares under the Gaussian-noise model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
np.set_printoptions(precision=6, suppress=True)

def poly_design(x, degree):
    x=np.asarray(x).ravel()
    return np.column_stack([x**j for j in range(degree+1)])

def gaussian_mle(X, y):
    X=np.asarray(X,float); y=np.asarray(y,float).ravel()
    w,_,rank,s=np.linalg.lstsq(X,y,rcond=None)
    pred=X@w
    e=y-pred
    sse=float(e@e)
    n=len(y)
    sigma2=sse/n
    beta=np.inf if sigma2==0 else 1/sigma2
    return w,pred,e,sse,sigma2,beta,rank,s

## 1. Previous-assignment data

The previous polynomial-regression assignment used `noisy_16.txt`. The notebook loads it directly from `../Lab 3/` so the experiment uses the same data rather than regenerating a different dataset.

In [ ]:
data_path=Path("../Lab 3/noisy_16.txt")
if not data_path.exists():
    raise FileNotFoundError("Place this notebook in Lab 5 inside the same repository as Lab 3.")
data=np.loadtxt(data_path)
x_prev,t_prev=data[:,0],data[:,1]
print("Shape:",data.shape)
print(data[:5])

In [ ]:
degree=3
Phi=poly_design(x_prev,degree)
w,pred,res,sse,sigma2,beta,rank,sv=gaussian_mle(Phi,t_prev)

print("w_ML =",w)
print("SSE =",sse)
print("sigma^2_ML =",sigma2)
print("beta_ML =",beta)
print("rank =",rank)

xg=np.linspace(x_prev.min(),x_prev.max(),400)
plt.figure(figsize=(9,5))
plt.scatter(x_prev,t_prev,s=18,label="Observed data")
plt.plot(xg,poly_design(xg,degree)@w,linewidth=2,label="MLE polynomial")
plt.xlabel("x"); plt.ylabel("t"); plt.title("Previous-assignment data: Gaussian MLE")
plt.legend(); plt.grid(alpha=.25); plt.show()

plt.figure(figsize=(9,4))
plt.axhline(0,linewidth=1); plt.scatter(x_prev,res,s=18)
plt.xlabel("x"); plt.ylabel("Residual"); plt.title("MLE residuals")
plt.grid(alpha=.25); plt.show()

## 2. Derivation

The likelihood is
\[
p(\mathbf t|\Phi,\mathbf w,\beta)=
\prod_{n=1}^N
\mathcal N(t_n|\mathbf w^T\phi_n,\beta^{-1}).
\]

Therefore
\[
\ln p=
\frac N2\ln\beta-\frac N2\ln(2\pi)
-\frac{\beta}{2}\sum_n(t_n-\mathbf w^T\phi_n)^2.
\]

Setting \(`\partial\ln p/\partial\mathbf w=0`\):
\[
\Phi^T\Phi\,\mathbf w_{ML}=\Phi^T\mathbf t
\]
and hence
\[
\boxed{\mathbf w_{ML}=(\Phi^T\Phi)^{-1}\Phi^T\mathbf t}.
\]

Setting the derivative with respect to \beta to zero gives
\[
\boxed{\beta_{ML}=\frac{N}{\sum_n(t_n-\hat t_n)^2}}.
\]

In code, `np.linalg.lstsq` is used instead of explicitly forming the inverse because it is more numerically stable.

In [ ]:
# Verify the analytical normal equation against lstsq
w_normal=np.linalg.solve(Phi.T@Phi,Phi.T@t_prev)
print("Normal-equation w:",w_normal)
print("Maximum coefficient difference:",
      np.max(np.abs(w_normal-w)))

## 3. Real data — Diabetes regression dataset

The scikit-learn diabetes dataset contains 442 observations, 10 numerical features and a continuous target. We fit
\[
\mathbf y=X\mathbf w+\epsilon,\qquad
\epsilon\sim\mathcal N(0,\sigma^2I).
\]
An intercept is included explicitly.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score

ds=load_diabetes()
X,y=ds.data,ds.target
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42)

Xtr=np.column_stack([np.ones(len(Xtr)),Xtr])
Xte=np.column_stack([np.ones(len(Xte)),Xte])

wr,ptr,etr,sser,sigmar,betar,rankr,svr=gaussian_mle(Xtr,ytr)
pte=Xte@wr

print("w_ML [intercept, coefficients] =",wr)
print("sigma^2_ML =",sigmar)
print("beta_ML =",betar)
print("Train RMSE =",np.sqrt(mean_squared_error(ytr,ptr)))
print("Test RMSE =",np.sqrt(mean_squared_error(yte,pte)))
print("Test R^2 =",r2_score(yte,pte))

plt.figure(figsize=(7,6))
plt.scatter(yte,pte,alpha=.75)
lims=[min(yte.min(),pte.min()),max(yte.max(),pte.max())]
plt.plot(lims,lims,linewidth=2)
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.title("Real data: MLE regression")
plt.grid(alpha=.25); plt.show()

plt.figure(figsize=(8,4))
plt.axhline(0,linewidth=1); plt.scatter(pte,yte-pte,alpha=.75)
plt.xlabel("Predicted"); plt.ylabel("Residual")
plt.title("Real-data test residuals")
plt.grid(alpha=.25); plt.show()

## 4. Interpretation and viva

**Previous data:** The polynomial coefficients are MLE estimates under Gaussian observation noise. The estimated variance measures the residual noise level.

**Real data:** The coefficient vector is the MLE of the linear regression parameters. Test RMSE and $R^2$ measure generalization and are separate from maximizing the training likelihood.

**Important:** MLE uses
\[
\sigma^2_{ML}=\frac{\mathrm{SSE}}{N},
\]
not the unbiased estimator \(\mathrm{SSE}/(N-p)\).

### Viva questions
1. Why does Gaussian MLE lead to least squares?
2. What does $\beta$ represent?
3. Why use $N$, not $N-p$, for MLE variance?
4. Why is `lstsq` preferable to an explicit inverse?
5. What happens when $X^TX$ is singular?
6. Difference between MLE and MAP?
7. Why can training likelihood be high while test performance is poor?
8. What changes if the noise covariance is not $\sigma^2I$?

### Complexity
For $N$ samples and $d$ parameters: forming $X^TX$ is $O(Nd^2)$, solving the $d\times d$ system is $O(d^3)$, prediction is $O(Nd)$, and storing $X$ is $O(Nd)$.